In [3]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.keys import Keys
import pandas as pd 
import random
import time

In [4]:
# 로또 사이트에서 당첨결과 엑셀 파일 다운로드

# Service 객체 생성
service = webdriver.chrome.service.Service(ChromeDriverManager().install())
# Service 객체를 webdriver.Chrome() 메서드에 전달
driver = webdriver.Chrome(service=service)
# Chrome 실행 파일 경로 설정
webdriver.chrome.service.DEFAULT_EXECUTABLE_PATH = '/Applications/Google Chrome.app/Contents/MacOS/Google Chrome'

url = 'https://dhlottery.co.kr/gameResult.do?method=byWin&wiselog=H_C_1_1'
driver.get(url)

# 다운받을 회차 범위
select = Select(driver.find_element(By.CSS_SELECTOR, "#drwNoStart"))
select.select_by_visible_text("1")
select = Select(driver.find_element(By.CSS_SELECTOR, "#drwNoEnd"))
# select.select_by_visible_text(str(end))
driver.find_element(By.CSS_SELECTOR, "#exelBtn").send_keys(Keys.ENTER)

print("파일 다운로드 완료!")
time.sleep(3)
driver.quit()

파일 다운로드 완료!


In [32]:
# 엑셀 파일을 열어서 데이터프레임으로 바꾸기
excel_file_path = "/Users/jongho/Downloads/lotto.xlsb"

df = pd.read_excel(excel_file_path, header=2)

new_df = df[['Unnamed: 1', 1, 2, 3, 4, 5, 6]]
new_df.columns = ['round', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6']

# print(new_df)

# 숫자별 당첨 확률 계산
round_start = min(new_df['round'])
round_end = max(new_df['round'])

counts = new_df.melt(id_vars=['round'])
number_prob_df = pd.DataFrame(counts['value'].value_counts().sort_index().reset_index().rename(columns={'value': 'number'}))

total_count = number_prob_df['count'].sum()
number_prob_df['prob'] = number_prob_df['count'] / total_count

# 숫자별 당첨 확률을 가중치로 반영해서 무작위 숫자 추출
random_numbers = [random.random() for _ in range(len(number_prob_df))]

number_prob_df['random'] = random_numbers
number_prob_df['random_1'] = number_prob_df['random'] * number_prob_df['prob']
number_prob_df['random_2'] = number_prob_df['random'] * (1 / number_prob_df['prob'])

# print(number_prob_df)

result_1 = number_prob_df.sort_values(by='random_1', ascending=False).head(6)
result_1 = result_1['number'].tolist()
result_1 = sorted(result_1)

result_2 = number_prob_df.sort_values(by='random_2', ascending=False).head(6)
result_2 = result_2['number'].tolist()
result_2 = sorted(result_2)


# 결과 출력
print(f"분석회차 : {round_start}회 ~ {round_end}회")
print(f"추출번호(과거   당첨 우선): {result_1}")
print(f"추출번호(과거 미당첨 우선): {result_2}")

분석회차 : 1회 ~ 1115회
추출번호(과거   당첨 우선): [3, 10, 28, 40, 44, 45]
추출번호(과거 미당첨 우선): [3, 10, 28, 30, 44, 45]
